# Session 12 — Why Quantum Optimal Transport?

**Author:** PPSP lab  **·**  **Date:** 2026-05-27  **·**  **Project:** Quantum Optimal Transport course  **·**  **Status:** 🔬 Teaching

## Purpose

S12 opens **Movement 4 (Quantum Optimal Transport)** by answering the question that has
quietly run through the whole course: *why isn't classical OT enough for density
matrices?* Three answers come together:

1. **The diagonal-collapse problem.** Two density matrices with the same diagonal in
   one basis are not the same state — and even commuting states have no *canonical*
   basis to read.
2. **Non-commutativity** is the operator-level source of the gap: when $[\rho_0, \rho_1]
   \neq 0$, no shared eigenbasis exists, so a classical reduction to "diagonals" is
   meaningless.
3. **Trevisan's taxonomy.** Multiple candidate definitions of quantum OT have been
   proposed (couplings, channels, dynamic, qubit-$W_1$); they are *not* all equivalent
   and capture different aspects.

We then **complete the dictionary** that has been growing since S1.

## 0. What you'll be able to do

- State the **diagonal-collapse problem** and demonstrate it on $|+\rangle\langle+|$ vs $I/2$.
- Build two density matrices that have **identical Z-basis diagonals** yet **do not commute**, and explain why classical OT cannot tell them apart.
- State the **diagonal-collapse consistency principle** that every candidate quantum OT must satisfy.
- Survey the **landscape of QOT formulations** (Trevisan, 2022).
- Read the **completed classical ↔ quantum dictionary**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ot

from qot_course import viz
from qot_course.infotheory.quantum import bures_distance, quantum_relative_entropy
from qot_course.quantum.density import density_matrix, maximally_mixed
from qot_course.quantum.states import KET_0, KET_PLUS, qubit_state
from qot_course.quantum_ot.preview import (
    commutativity_norm, diagonal_in_computational_basis, same_diagonal,
)
from qot_course.transport.discrete import cityblock_cost

np.random.seed(42)
viz.use_course_style()

## 1. The setting

Classical optimal transport takes two **probability vectors** $a, b \in \Delta^n$ and a
cost matrix $C$ and returns a distance. Density matrices live in a richer space: a
density matrix $\rho$ on $\mathbb{C}^d$ is a $d \times d$ Hermitian positive-semidefinite
operator with unit trace.

Two questions arise immediately:

> **Naive approach.** Take the diagonal $p_i = \rho_{ii}$ as a probability vector and do
> classical OT. *Doesn't this work?*

> **Problem.** The diagonal depends on the basis. Different bases give *different*
> answers — and only the spectrum (eigenvalues, basis-independent) is intrinsic to the
> state. We need a notion of OT that respects the operator structure, not just one
> shadow of it.

We will now expose the gap concretely.

## 2. The canonical example: $|+\rangle\langle+|$ versus $I/2$

These two qubit states have **identical** measurement statistics in the computational
($Z$) basis: $\langle 0|\rho|0\rangle = \langle 1|\rho|1\rangle = 1/2$ for both. But

- $|+\rangle\langle+|$ is **pure** (entropy 0), a coherent superposition;
- $I/2$ is **maximally mixed** (entropy 1 bit), a classical coin flip.

A measurement in the $X$ basis discriminates them perfectly: $|+\rangle\langle+|$ gives
$|+\rangle$ with probability $1$; $I/2$ gives $|+\rangle$ or $|-\rangle$ with
probability $1/2$ each. **Same statistics in one basis, opposite statistics in another.**

In [ ]:
plus  = density_matrix(KET_PLUS)
mixed = maximally_mixed(2)

print(f"diagonal of |+><+| (Z basis): {diagonal_in_computational_basis(plus).tolist()}")
print(f"diagonal of I/2    (Z basis): {diagonal_in_computational_basis(mixed).tolist()}")
print(f"same Z-diagonal?              {same_diagonal(plus, mixed)}")
print()

# Classical OT applied to the (identical) Z-diagonals returns ZERO.
cost = cityblock_cost([0, 1], [0, 1])
w1_diagonals = ot.emd2(
    diagonal_in_computational_basis(plus),
    diagonal_in_computational_basis(mixed),
    cost,
)
print(f"W_1 on Z-diagonals     = {w1_diagonals:.4f}   ← classical OT says they are identical")
print()

# But the actual states are very different. Bures (S7) and Umegaki KL (S7) both see it.
print(f"Bures distance         = {bures_distance(plus, mixed):.4f}")
print(f"Umegaki S(|+><+| || I/2) = {quantum_relative_entropy(plus, mixed):.4f} bit")

**Read the output.** Classical OT on the diagonals returns **zero** — it cannot see
the difference. But every honest quantum-aware metric (Bures, Umegaki relative
entropy) returns a strictly positive value. **The diagonal is not enough.** This single
example is the entire motivation for M4.

## 3. Even non-commuting states can share a diagonal

The $|+\rangle\langle+|$ vs $I/2$ example was actually a *commuting* pair (since $I/2$
commutes with everything). The deeper issue is that **non-commuting** states can also
have identical diagonals. Take

- $\rho_X = |+\rangle\langle+|$  (Bloch vector $(1, 0, 0)$)
- $\rho_Y = |+i\rangle\langle+i|$ with $|+i\rangle = \tfrac{1}{\sqrt 2}(|0\rangle + i|1\rangle)$ (Bloch vector $(0, 1, 0)$)

Both are pure states on the equator of the Bloch sphere, so both have Z-diagonal
$(1/2, 1/2)$ — but their off-diagonal coherences differ in *phase*.

In [ ]:
plus_x = density_matrix(KET_PLUS)                                # Bloch (1, 0, 0)
plus_y = density_matrix(qubit_state(theta=np.pi/2, phi=np.pi/2)) # Bloch (0, 1, 0)

print("rho_X (|+><+|, Bloch X-axis):")
print(np.round(plus_x, 3))
print()
print("rho_Y (|+i><+i|, Bloch Y-axis):")
print(np.round(plus_y, 3))
print()
print(f"same Z-diagonal?                  {same_diagonal(plus_x, plus_y)}")
print(f"commutator norm  ||[rho_X, rho_Y]|| = {commutativity_norm(plus_x, plus_y):.4f}   (>0 ⇒ no shared eigenbasis)")
print(f"Bures distance  d_B(rho_X, rho_Y) = {bures_distance(plus_x, plus_y):.4f}")

**Read the output.** Both states have Z-diagonal $(1/2, 1/2)$, so the naive classical
reduction calls them identical. But their commutator is non-zero, meaning **they cannot
be simultaneously diagonalised** — there is no shared basis in which both look like a
classical probability vector. The Bures distance is positive: an *intrinsic* quantum
metric recognises the difference.

Classical OT is **blind** in this regime by construction. Quantum OT is what we will
build to see through.

## 4. The diagonal-collapse consistency principle

A candidate "quantum OT" must satisfy a **consistency requirement**: whenever two density
matrices commute (share an eigenbasis), the candidate must reduce to classical OT on the
common eigenbasis.

This is the **diagonal-collapse principle**: classical OT is the *commuting* boundary of
quantum OT. The Trevisan (2022) survey uses this as a baseline test: any proposed
formulation must collapse to classical OT for diagonal states, otherwise it cannot be
called a quantum analogue. We verify the principle visually here; S13's SDP will satisfy
it by construction.

In [ ]:
# Two diagonal density matrices in the Z basis (= commuting classical states).
p_vec = np.array([0.6, 0.4])
q_vec = np.array([0.1, 0.9])
rho_p = np.diag(p_vec).astype(complex)
rho_q = np.diag(q_vec).astype(complex)

print(f"rho_p, rho_q both diagonal in Z basis:")
print(f"  diag(rho_p) = {diagonal_in_computational_basis(rho_p).tolist()}")
print(f"  diag(rho_q) = {diagonal_in_computational_basis(rho_q).tolist()}")
print(f"  commutator norm = {commutativity_norm(rho_p, rho_q):.2e}   ← zero, they commute")
print()

# Classical OT on the diagonals: this is the *correct* answer for the commuting case.
classical_w1 = ot.emd2(p_vec, q_vec, cost)
print(f"Classical W_1 on the diagonals = {classical_w1:.4f}")
print()
print("The diagonal-collapse principle: any candidate quantum OT MUST return this same")
print("number for these two commuting density matrices. In S13 the quantum coupling SDP")
print("will satisfy this by construction (the optimal coupling becomes diagonal).")

## 5. Trevisan's taxonomy — the QOT landscape

Quantum OT is not a single object. The field has produced several non-equivalent
formulations, each capturing different aspects of "transport on density matrices":

| Formulation | Idea | Course session |
|---|---|---|
| **Quantum couplings (SDP)** | Replace classical couplings $P$ by bipartite density matrices $\rho_{AB}$ with partial-trace marginals; minimise $\mathrm{tr}(C\,\rho_{AB})$ (De Palma–Trevisan, 2021). | **S13** |
| **Entropic / quantum Sinkhorn** | Add a von-Neumann-entropy regulariser; use matrix exponentials + partial traces as the fixed point (Peyré, Cuturi, Gerolin et al.). | **S14** |
| **Dynamic / Carlen–Maas** | Replace the continuity equation by a Lindblad-type generator; define a quantum Wasserstein on the dynamics of density matrices. | (S14, brief) |
| **Channel-based** | Define a Wasserstein-like distance via the action of CPTP channels (no genuine coupling — uses the channel's Stinespring dilation). | (S16) |
| **Qubit-$W_1$** | Lipschitz dual specialised to multi-qubit systems with a Hamming-type metric (De Palma, Marvian, Trevisan, Lloyd, 2021). | (S16) |

**Different formulations, different theorems.** No single "the" quantum OT yet. This
course uses the **coupling SDP** (S13) as the principal object — it is the most direct
analogue of the Kantorovich LP and the one that admits both an entropic regularisation
(quantum Sinkhorn, S14) and a clean information-geometric reading (the Amari bridge
survives).

## 6. The completed classical ↔ quantum dictionary

Eleven sessions later, the full dictionary that has run through the course:

| Classical | Quantum | First seen |
|-----------|---------|-----------|
| probability vector $a \in \Delta^n$ | density matrix $\rho \in \mathcal{S}(\mathcal{H})$ | S3 |
| marginal | partial trace | S4 |
| product distribution | product state | S4 |
| Markov kernel | quantum channel (CPTP map) | S4 |
| Shannon entropy $H(p)$ | von Neumann entropy $S(\rho)$ | S5 / S7 |
| KL divergence $D(p\|q)$ | Umegaki relative entropy $S(\rho\|\sigma)$ | S7 |
| mutual information $I(X;Y) \le \min(H_X, H_Y)$ | quantum MI $I(A{:}B)$ (up to $2\log d$) | S7 |
| conditional entropy $H(X|Y) \ge 0$ | quantum CMI $S(A|B)$ — **can be negative** | S7 |
| Fisher–Rao metric | Bures / quantum Fisher information | S6 / S7 |
| transfer entropy $\mathrm{TE}$ | (genuinely open — quantum TE not unique) | S5 / S15 |
| **transport plan $P \in T(a, b)$** | **quantum coupling $\rho_{AB}$ with marginals** | S13 |
| **Kantorovich LP** | **quantum OT SDP** | S13 |
| **transportation polytope $T(a, b)$** | **set of quantum couplings** | S13 |
| Birkhoff polytope (Birkhoff–von Neumann) | (assignment-style quantum couplings) | S15 mention |
| **Wasserstein-$p$ distance $W_p$** | **quantum Wasserstein** | S13 |
| **1-D quantile closed form** | no analogue — operators don't sort | — |
| Bures–Wasserstein on Gaussians | Bures distance on density matrices (S7) | **S11 (the bridge)** |
| McCann displacement geodesic | quantum displacement interpolation | S14 |
| **Sinkhorn algorithm** | **quantum Sinkhorn (matrix exp + partial trace)** | S14 |
| **Sinkhorn = KL projection (Amari)** | **quantum Sinkhorn = Umegaki projection** | S14 |
| Benamou–Brenier / Otto calculus | quantum Riemannian metric (Carlen–Maas) | S14 |

**This is the spine of the course.** Every row is a structural correspondence; many were
proven theorem-by-theorem along the way, some remain open (transfer entropy, qubit-$W_1$
dynamics).

## 7. Your turn

1. **Diagonal-collapse, your own example.** Pick two commuting Hermitian density
   matrices (e.g. both diagonal in a non-trivial basis) and verify that their commutator
   is zero. State what *classical OT on the diagonal* would give. *(In S13 we will run the
   quantum SDP and confirm it matches.)*
2. **Coherence-distance scaling.** Parametrise a family $\rho(\theta) = (1 - \theta)|+\rangle\langle+| + \theta\,I/2$
   for $\theta \in [0, 1]$. Plot the Bures distance $d_B(\rho(0), \rho(\theta))$ as a
   function of $\theta$. How does it relate to the *purity* of $\rho(\theta)$?
3. **Same-spectrum, different states.** Build two density matrices with the **same
   spectrum** (e.g. eigenvalues $(0.7, 0.3)$) but different eigenbases (e.g. one in $Z$
   basis, one in $X$ basis). Show that they have **different** Z-diagonals but **the
   same** von Neumann entropy. Which OT formulation can distinguish them?

## Conclusions & references

- The **naive classical reduction** of a density matrix to its diagonal is
  **basis-dependent** and *loses information*. Two states with identical Z-diagonals are
  generically different.
- **Non-commutativity** removes even the option of choosing a shared basis: no canonical
  "diagonal" exists. Classical OT is structurally blind in this regime.
- The **diagonal-collapse principle** is the consistency requirement for any quantum OT:
  it must reduce to classical OT on commuting states.
- **Trevisan's taxonomy** lists multiple non-equivalent formulations; this course follows
  the **coupling-SDP** path (S13) and its entropic regularisation (S14, *quantum
  Sinkhorn*).
- The **classical ↔ quantum dictionary** is essentially complete now — only the explicit
  QOT entries remain to be filled in (S13, S14, S15).
- **Next (S13 — Coupling QOT = a semidefinite program):** the actual definition.
  Quantum couplings, the SDP, `cvxpy`, validation against Bures.

**References:** D. Trevisan, "Optimal transport methods for quantum systems",
arXiv:2202.02091 (2022); G. De Palma, D. Trevisan, "Quantum optimal transport with
quantum channels", Ann. Henri Poincaré 22, 3199 (2021); G. De Palma, M. Marvian, D.
Trevisan, S. Lloyd, "The quantum Wasserstein distance of order 1", IEEE Trans. Inf.
Theory 67, 6627 (2021); E. Carlen, J. Maas, "An analog of the 2-Wasserstein metric in
non-commutative probability under which the Fermionic Fokker-Planck equation is
gradient flow for the entropy", Comm. Math. Phys. 331, 887 (2014). Previous:
`notebooks/03_ClassicalOptimalTransport/04_gaussians.ipynb`.